
# Extended Financial Inclusion Indicators — FinScope Rwanda 2024

## Objective
Identify and analyse additional indicators of financial inclusion using the FinScope 2024 Rwanda dataset.

## Areas to investigate
1. Formal savings participation
2. SACCO participation
3. Active mobile-money usage
4. Insurance coverage
5. Formal credit access
6. Financial resilience
7. Barriers to financial access
8. Digital financial-service activity
9. Informal savings-group participation
10. Rotating savings and credit groups
11. Informal saving mechanisms
12. Informal borrowing and mutual support

## Method
- Verify each indicator against the questionnaire and variable labels.
- Document its source variable, definition and response categories.
- Calculate weighted estimates using `pop_wt` where appropriate.
- Compare districts only where data quality and sample sizes support it.
- Save validated outputs under `data/processed/`.

**Important:** These are candidate concepts. Some may overlap, and some may not be measurable from the available variables.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent
data_path = project_root / "data" / "datasets" / "FinScope2024Rwanda.dta"

df = pd.read_stata(data_path, convert_categoricals=True)

print(f"Dataset shape: {df.shape}")
print(f"Respondents: {len(df):,}")
print(f"Project root: {project_root}")
print(f"Weight variable available: {'pop_wt' in df.columns}")


Dataset shape: (13994, 2442)
Respondents: 13,994
Project root: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project
Weight variable available: True


In [2]:
# Read the original Stata metadata without loading the dataset again
try:
    import pyreadstat

    _, metadata = pyreadstat.read_dta(
        data_path,
        metadataonly=True
    )

    labels = metadata.column_names_to_labels

    print("Variables with labels:", sum(bool(v) for v in labels.values()))
    print("Total variable names:", len(metadata.column_names))

    # Search variable labels and names for candidate indicators
    keywords = [
        "saving", "savings", "sacco", "cooperative", "group",
        "informal", "rotating", "mobile money", "transaction",
        "insurance", "insured", "credit", "loan", "borrow",
        "emergency", "unexpected", "barrier", "afford",
        "distance", "documentation", "digital", "friend",
        "family", "relative", "community", "transfer"
    ]

    matches = []

    for col in metadata.column_names:
        label = labels.get(col, "") or ""
        searchable = f"{col} {label}".lower()
        found = [kw for kw in keywords if kw in searchable]

        if found:
            matches.append({
                "Variable": col,
                "Label": label,
                "Matched keywords": ", ".join(found)
            })

    variable_matches = pd.DataFrame(matches)

    print("Candidate variables found:", len(variable_matches))

    if not variable_matches.empty:
        display(variable_matches)
    else:
        print("No candidates found. We will inspect the questionnaire/dictionary.")

except ImportError:
    print("pyreadstat is not installed in this environment.")

Variables with labels: 2442
Total variable names: 2442
Candidate variables found: 954


,Variable,Label,Matched keywords
0,e3,"E3. Before you buy something, you carefully co...",afford
1,e9b,E9b.Have you set aside emergency/rainy-day fun...,emergency
2,e10a2,E10a2. Have 4 or more credit commitments,credit
3,e10a3,E10a3. Your credit/borrowing commitments are a...,"credit, borrow"
4,e10a4,E10a4. You have enough money left for savings ...,"saving, savings"
...,...,...,...
949,d1a13,D1a13. What is the distance from here to the n...,distance
950,d1a14,D1a14. What is the distance from here to the n...,"mobile money, distance"
951,d1b14,D1b14. How long would it take you to get to yo...,mobile money
952,d1a15,D1a15. How long would it take you to get to yo...,insurance


In [3]:

print("Total variables:", len(df.columns))
print("Matches found:", len(matches))
print("DataFrame label metadata:", list(
    getattr(df, "attrs", {}).keys()
))

if matches:
    variable_matches = (
        pd.DataFrame(matches)
        .sort_values("Variable")
        .reset_index(drop=True)
    )
    display(variable_matches.head(100))
else:
    print("No matches found using the current keyword search.")
    print("\nFirst 30 variable names:")
    print(df.columns[:30].tolist())

    print("\nExample columns and their values:")
    for col in df.columns[:10]:
        print(f"\n{col}: {df[col].dropna().head(3).tolist()}")


Total variables: 2442
Matches found: 954
DataFrame label metadata: []


,Variable,Label,Matched keywords
0,c13d,C13d. Has your household received a direct cas...,transfer
1,c7c_1,c7c_1. Loan from a financial institution,loan
2,c7c_2,c7c_2. Borrow from money lender,borrow
3,c7c_3,"c7c_3. Borrow from savings group, tontine","saving, savings, group, borrow"
4,c7c_4,c7c_4. Borrowed from employer,borrow
...,...,...,...
95,d1a8,D1a8. What is the distance from here to the ne...,"sacco, distance"
96,d1b14,D1b14. How long would it take you to get to yo...,mobile money
97,d1b15,D1b15. What is the distance from here to the n...,"insurance, distance"
98,d1b7,D1b7. How long would it take you to get to you...,sacco


In [4]:
import re

pattern = (
    r"sacco|cooperative|rotating savings|"
    r"village savings|savings club|informal group|"
    r"group savings|saving.*group|group.*saving|"
    r"stokvel|rosca"
)

results = []

for col in metadata.column_names:
    label = labels.get(col, "") or ""

    if re.search(pattern, f"{col} {label}", re.IGNORECASE):
        results.append((col, label))

print(f"Matching variables: {len(results)}")

for col, label in results[:30]:
    print(f"{col}: {label}")

if len(results) > 30:
    print(f"... {len(results) - 30} additional matches omitted.")

Matching variables: 183
e18_3: e18_3. Borrowed from saving group
e18_9: e18_9. Used savings from a saving group
qf1_06: qf1_06. Ever used - Umurenge SACCO
qf1_10: qf1_10. Ever used - Savings group that you save with or borrow from
qf2_06: qf2_06. Offer services electronically - Umurenge Sacco
qf2_10: qf2_10. Offer services electronically - Savings group
qf3_06: qf3_06. Account registered in own name - Umurenge SACCO
qf3_10: qf3_10. Account registered in own name - Savings group that you save with or bor
qf4_06: qf4_06. Do you currently use - Umurenge SACCO
qf4_10: qf4_10. Do you currently use - Savings group that you save with or borrow from
qf5_06_1: qf5_06_1. Umurenge SACCO-Deposited or withdrew money
qf5_06_2: qf5_06_2. Umurenge SACCO-Bought airtime top-ups or paid a mobile phone bill
qf5_06_3: qf5_06_3. Umurenge SACCO-Paid school fees
qf5_06_4: qf5_06_4. Umurenge SACCO-Paid a bill for medical treatment, housing, trash colle
qf5_06_5: qf5_06_5. Umurenge SACCO-Paid a tax, fine, fee, 

In [5]:
check_vars = ["qf1_06", "qf4_06", "qf1_10", "qf4_10", "e18_3", "e18_9"]

for col in check_vars:
    print(f"\n{col}: {labels.get(col, '')}")
    print("Non-missing:", df[col].notna().sum())
    print(df[col].value_counts(dropna=False).head(8).to_dict())


qf1_06: qf1_06. Ever used - Umurenge SACCO
Non-missing: 13994
{'No': 9822, 'Yes': 4172}

qf4_06: qf4_06. Do you currently use - Umurenge SACCO
Non-missing: 4172
{nan: 9822, 'Yes': 3383, 'No': 789}

qf1_10: qf1_10. Ever used - Savings group that you save with or borrow from
Non-missing: 13994
{'Yes': 7026, 'No': 6968}

qf4_10: qf4_10. Do you currently use - Savings group that you save with or borrow from
Non-missing: 7026
{nan: 6968, 'Yes': 6655, 'No': 371}

e18_3: e18_3. Borrowed from saving group
Non-missing: 11219
{'No': 9677, nan: 2775, 'Yes': 1542}

e18_9: e18_9. Used savings from a saving group
Non-missing: 11219
{'No': 9769, nan: 2775, 'Yes': 1450}


In [6]:
weight = "pop_wt"
district = "a2"

indicator_specs = {
    "Ever used Umurenge SACCO (%)": ("qf1_06", "Yes"),
    "Currently uses Umurenge SACCO (%)": ("qf4_06", "Yes"),
    "Ever used a savings group (%)": ("qf1_10", "Yes"),
    "Currently uses a savings group (%)": ("qf4_10", "Yes"),
}

# National weighted estimates
national_results = []

for indicator, (col, positive_value) in indicator_specs.items():
    valid = df[col].notna() & df[weight].notna()
    numerator = df.loc[
        valid & df[col].eq(positive_value), weight
    ].sum()
    denominator = df.loc[valid, weight].sum()

    national_results.append({
        "Indicator": indicator,
        "Weighted estimate (%)": 100 * numerator / denominator,
        "Valid respondents": int(valid.sum())
    })

national_indicators = pd.DataFrame(national_results)
display(national_indicators.round(2))

,Indicator,Weighted estimate (%),Valid respondents
0,Ever used Umurenge SACCO (%),26.10,13994
1,Currently uses Umurenge SACCO (%),81.43,4172
2,Ever used a savings group (%),48.15,13994
3,Currently uses a savings group (%),94.92,7026


In [7]:
current_use_specs = {
    "Currently uses Umurenge SACCO (%)": ("qf1_06", "qf4_06"),
    "Currently uses a savings group (%)": ("qf1_10", "qf4_10"),
}

results = []

for indicator, (ever_col, current_col) in current_use_specs.items():
    # Full population denominator, using the survey weight
    current_yes = df[current_col].eq("Yes")

    numerator = df.loc[current_yes, "pop_wt"].sum()
    denominator = df.loc[df["pop_wt"].notna(), "pop_wt"].sum()

    results.append({
        "Indicator": indicator,
        "Full-population weighted estimate (%)": 100 * numerator / denominator,
        "Current-use Yes responses": int(current_yes.sum())
    })

display(pd.DataFrame(results).round(2))

,Indicator,Full-population weighted estimate (%),Current-use Yes responses
0,Currently uses Umurenge SACCO (%),21.26,3383
1,Currently uses a savings group (%),45.71,6655


In [8]:
from pathlib import Path
import pandas as pd

weight_col = "pop_wt"
district_col = "a2"

# Create explicit full-population binary indicators.
# For current use, skipped respondents are non-users because
# the current-use question is routed from the "ever used" question.
df["sacco_ever_used"] = df["qf1_06"].eq("Yes").astype(int)
df["sacco_current_use"] = df["qf4_06"].eq("Yes").astype(int)

df["savings_group_ever_used"] = df["qf1_10"].eq("Yes").astype(int)
df["savings_group_current_use"] = df["qf4_10"].eq("Yes").astype(int)

indicator_cols = {
    "Ever used Umurenge SACCO (%)": "sacco_ever_used",
    "Currently uses Umurenge SACCO (%)": "sacco_current_use",
    "Ever used a savings group (%)": "savings_group_ever_used",
    "Currently uses a savings group (%)": "savings_group_current_use",
}

rows = []

for district_name, group in df.groupby(district_col, observed=True):
    valid = group[weight_col].notna() & group[weight_col].gt(0)
    g = group.loc[valid]

    row = {
        "District": district_name,
        "Valid respondents": len(g),
    }

    denominator = g[weight_col].sum()

    for label, col in indicator_cols.items():
        weighted_yes = (g[col] * g[weight_col]).sum()
        row[label] = 100 * weighted_yes / denominator

    rows.append(row)

district_indicators = pd.DataFrame(rows).sort_values("District")

output_path = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "district_extended_indicators.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)
district_indicators.to_csv(output_path, index=False)

print(f"Districts included: {len(district_indicators)}")
print(f"Saved to: {output_path}")
display(district_indicators.round(2))

Districts included: 30
Saved to: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/district_extended_indicators.csv


,District,Valid respondents,Ever used Umurenge SACCO (%),Currently uses Umurenge SACCO (%),Ever used a savings group (%),Currently uses a savings group (%)
29,Bugesera,560,28.28,20.87,49.39,46.35
21,Burera,416,25.18,20.85,52.20,49.17
19,Gakenke,447,28.83,24.57,51.14,49.99
1,Gasabo,848,24.98,22.01,42.38,39.59
25,Gatsibo,400,21.90,17.59,57.51,55.41
22,Gicumbi,512,34.15,30.86,46.94,46.13
4,Gisagara,400,30.55,24.10,66.34,62.60
6,Huye,480,19.01,15.53,51.95,50.37
10,Kamonyi,623,24.45,21.31,51.15,49.38
11,Karongi,399,32.75,25.67,54.75,52.75


In [9]:
pattern = r"mobile money.*(send|receive|pay|transaction|use|used)|" \
          r"(send|receive|pay|transaction).*mobile money"

for col in metadata.column_names:
    label = labels.get(col, "") or ""
    if re.search(pattern, label, re.IGNORECASE):
        print(f"{col}: {label}")

qf5_05_5: qf5_05_5. Mobile money operator-Paid a tax, fine, fee, or other payment to the g
qf5_05_6: qf5_05_6. Mobile money operator-Sent money to, or received money from someone
qf5_05_7: qf5_05_7. Mobile money operator-Received money from the government
qf5_05_8: qf5_05_8. Mobile money operator-Received wages from an employer
qf5_05_9: qf5_05_9. Mobile money operator-Made insurance payments or received claims on in
qf5_05_14: qf5_05_14. Mobile money operator-Made pension payment or retirement savings
qf5_05_15: qf5_05_15. Mobile money operator-Never used any service in this institution
qf6_05_3: qf6_05_3.Mobile money operator-Poor service received at a branch / agent / custo
qf6_05_9: qf6_05_9.Mobile money operator-Unclear transaction charges/fees
qf6_05_12: qf6_05_12.Mobile money operator-Sending to a wrong number
qf6_05_15: qf6_05_15.Mobile money operator-Delayed payments
h8a_4: h8a_4. Mobile money-In the past 6 months, have you been refused a loan
p2_01_10: p2. How do you normally

In [10]:
check_vars = ["qf5_05_6", "qf5_05_7", "qf5_05_8", "p2_01_10"]

for col in check_vars:
    print(f"\n{col}: {labels.get(col, '')}")
    print(df[col].value_counts(dropna=False).head(6).to_dict())


qf5_05_6: qf5_05_6. Mobile money operator-Sent money to, or received money from someone
{nan: 9079, 'Yes': 2578, 'No': 2337}

qf5_05_7: qf5_05_7. Mobile money operator-Received money from the government
{nan: 9079, 'No': 4792, 'Yes': 123}

qf5_05_8: qf5_05_8. Mobile money operator-Received wages from an employer
{nan: 9079, 'No': 4777, 'Yes': 138}

p2_01_10: p2. How do you normally pay: Food/drink - Mobile money
{'No': 11357, 'Yes': 1452, nan: 1185}


In [11]:
pattern = r"insurance|insured|insurance scheme|insurance coverage|health cover|premium"

results = []

for col in metadata.column_names:
    label = labels.get(col, "") or ""
    if re.search(pattern, label, re.IGNORECASE):
        results.append((col, label))

print(f"Insurance-related variables: {len(results)}")

for col, label in results[:20]:
    print(f"{col}: {label}")

if len(results) > 20:
    print(f"... {len(results) - 20} more omitted.")

Insurance-related variables: 168
e18_8: e18_8. Used savings held at a bank/Insurance provider/Retailer
e18_19: e18_19. Education insurance
qf1_07: qf1_07. Ever used - Insurance provider
qf3_07: qf3_07. Account registered in own name - Insurance provider
qf4_07: qf4_07. Do you currently use - Insurance provider
qf5_01_9: qf5_01_9. Commercial bank-Made insurance payments or received claims on insuranc
qf5_02_9: qf5_02_9. Digital bank-Made insurance payments or received claims on insurance
qf5_03_9: qf5_03_9. Microfinance institution (non-umurenge)-Made insurance payments or rec
qf5_04_9: qf5_04_9. Non-Deposit Taking Lending Financial (NDFI) -Made insurance payments o
qf5_05_9: qf5_05_9. Mobile money operator-Made insurance payments or received claims on in
qf5_06_9: qf5_06_9. Umurenge SACCO-Made insurance payments or received claims on insurance
qf5_07_1: qf5_07_1. Insurance provider-Deposited or withdrew money
qf5_07_2: qf5_07_2. Insurance provider-Bought airtime top-ups or paid a mobil

In [12]:
check_vars = ["qf1_07", "qf4_07", "e18_19"]

for col in check_vars:
    print(f"\n{col}: {labels.get(col, '')}")
    print("Non-missing:", int(df[col].notna().sum()))
    print(df[col].value_counts(dropna=False).head(8).to_dict())

print("\nOther direct insurance coverage questions:")
for col in metadata.column_names:
    label = labels.get(col, "") or ""
    if re.search(r"covered by insurance|have insurance|insurance policy|type of insurance", label, re.I):
        print(f"{col}: {label}")


qf1_07: qf1_07. Ever used - Insurance provider
Non-missing: 13994
{'No': 13796, 'Yes': 198}

qf4_07: qf4_07. Do you currently use - Insurance provider
Non-missing: 198
{nan: 13796, 'Yes': 175, 'No': 23}

e18_19: e18_19. Education insurance
Non-missing: 11219
{'No': 11212, nan: 2775, 'Yes': 7}

Other direct insurance coverage questions:
cc5_1_9: cc5_1_9. Heavy rain and floods:Claimed from an insurance policy
cc5_2_9: cc5_2_9. Storm:Claimed from an insurance policy
cc5_3_9: cc5_3_9. Drought/ water scarcity:Claimed from an insurance policy
cc5_4_9: cc5_4_9. Extreme heat/heatwaves:Claimed from an insurance policy
cc5_5_9: cc5_5_9. Bush burning/wildfires:Claimed from an insurance policy
cc5_6_9: cc5_6_9. Pest infestation:Claimed from an insurance policy
cc5_7_9: cc5_7_9. Erosion:Claimed from an insurance policy
cc5_8_9: cc5_8_9. Other:Claimed from an insurance policy
cc7_9: Claim from an insurance policy


In [13]:
pattern = r"credit provider|loan provider|borrowed from|loan from|credit from|"
pattern += r"commercial bank|microfinance institution|umurenge sacco"

results = []

for col in metadata.column_names:
    label = labels.get(col, "") or ""
    if re.search(pattern, label, re.IGNORECASE):
        results.append((col, label))

print(f"Matching variables: {len(results)}")

for col, label in results[:25]:
    print(f"{col}: {label}")

if len(results) > 25:
    print(f"... {len(results) - 25} more omitted.")

Matching variables: 226
e18_1: e18_1. Borrowed from a bank or other formal institution
e18_2: e18_2. Borrowed from money lender (loan shark)
e18_3: e18_3. Borrowed from saving group
e18_5: e18_5. Borrowed from family/friends/community/church/mosque
e18_6: e18_6. Took a loan from a shopkeeper
e18_7: e18_7. Borrowed from a digital app-based lender e.g., Mokash
qf1_01: qf1_01. Ever used - Commercial bank
qf1_03: qf1_03. Ever used - Microfinance institution (non-umurenge)
qf1_06: qf1_06. Ever used - Umurenge SACCO
qf2_01: qf2_01. Offer services electronically - Commercial bank
qf2_03: qf2_03. Offer services electronically - Microfinance institution(non-umurenge)
qf2_06: qf2_06. Offer services electronically - Umurenge Sacco
qf3_01: qf3_01. Account registered in own name - Commercial bank
qf3_03: qf3_03. Account registered in own name - Microfinance institution (non-umurenge)
qf3_06: qf3_06. Account registered in own name - Umurenge SACCO
qf4_01: qf4_01. Do you currently use - Commercial ba

In [14]:
check_vars = ["e18_1", "e18_2", "e18_3", "e18_5", "e18_6", "e18_7"]

for col in check_vars:
    print(f"\n{col}: {labels.get(col, '')}")
    print("Non-missing:", int(df[col].notna().sum()))
    print(df[col].value_counts(dropna=False).head(5).to_dict())


e18_1: e18_1. Borrowed from a bank or other formal institution
Non-missing: 11219
{'No': 10411, nan: 2775, 'Yes': 808}

e18_2: e18_2. Borrowed from money lender (loan shark)
Non-missing: 11219
{'No': 11174, nan: 2775, 'Yes': 45}

e18_3: e18_3. Borrowed from saving group
Non-missing: 11219
{'No': 9677, nan: 2775, 'Yes': 1542}

e18_5: e18_5. Borrowed from family/friends/community/church/mosque
Non-missing: 11219
{'No': 11057, nan: 2775, 'Yes': 162}

e18_6: e18_6. Took a loan from a shopkeeper
Non-missing: 11219
{'No': 11194, nan: 2775, 'Yes': 25}

e18_7: e18_7. Borrowed from a digital app-based lender e.g., Mokash
Non-missing: 11219
{'No': 11180, nan: 2775, 'Yes': 39}


In [15]:
pattern = r"unexpected expense|emergency fund|rainy.day|financial shock|"
pattern += r"come up with money|raise money|cope with|emergency"

results = []

for col in metadata.column_names:
    label = labels.get(col, "") or ""
    if re.search(pattern, label, re.IGNORECASE):
        results.append((col, label))

print(f"Matching variables: {len(results)}")

for col, label in results[:20]:
    print(f"{col}: {label}")

if len(results) > 20:
    print(f"... {len(results) - 20} more omitted.")

Matching variables: 2
e9b: E9b.Have you set aside emergency/rainy-day funds that would cover your expenses
cc3:  cc3- Did you put plans or measures in place to cope with problem from...?


In [16]:
for col in ["e9b", "cc3"]:
    print(f"\n--- {col}: {labels.get(col, 'No label')} ---")
    print("Data type:", df[col].dtype)
    print("Non-missing:", df[col].notna().sum())
    print("Responses:")
    print(df[col].value_counts(dropna=False).head(15))


--- e9b: E9b.Have you set aside emergency/rainy-day funds that would cover your expenses ---
Data type: category
Non-missing: 13994
Responses:
e9b
No     10976
Yes     3018
Name: count, dtype: int64

--- cc3:  cc3- Did you put plans or measures in place to cope with problem from...? ---
Data type: str
Non-missing: 13994
Responses:
cc3
    13994
Name: count, dtype: int64


In [17]:
# Emergency/rainy-day funds indicator
valid = df["e9b"].notna() & df["pop_wt"].notna()

yes = df.loc[valid, "e9b"].eq("Yes")

weighted_pct = (
    df.loc[valid & yes, "pop_wt"].sum()
    / df.loc[valid, "pop_wt"].sum()
) * 100

print(f"Valid respondents: {valid.sum():,}")
print(f"Weighted with emergency funds: {weighted_pct:.2f}%")
print(f"Weighted without emergency funds: {100 - weighted_pct:.2f}%")

Valid respondents: 13,994
Weighted with emergency funds: 20.78%
Weighted without emergency funds: 79.22%


In [18]:
import re

pattern = (
    r"why.*(don't|do not|not).*"
    r"(account|bank|save|borrow|insurance)|"
    r"barrier|reason.*not.*(use|have|open)|"
    r"too expensive|too far|lack of money|"
    r"lack of trust|documentation"
)

matches = [
    (col, labels.get(col, ""))
    for col in metadata.column_names
    if re.search(pattern, labels.get(col, ""), re.IGNORECASE)
]

print(f"Matching variables: {len(matches)}")
for col, label in matches[:25]:
    print(f"{col}: {label}")

Matching variables: 12
g9_12: g9. It is too expensive
h7: H7. What is the main reason why you have not borrowed money?
k8a: K8a. What are the main reasons why you do not use your transaction account, more
k8a_1: k8a_1. Using my account / card is too expensive
k8b_1: k8b_1. Using my account / card is too expensive
k17: K17. What is the main reason why you do not have a bank account?
k2_7: K2.7 What is the main reason why do you not have an MFI account?
k3_12: K3.12 What is the main reason why do you not have a bank account?
f14: F14. What is the main reason why do you not use mobile money?
l8_2: l9_2. They are too expensive
m9_4: m9_4. The market is too far away
c7e_11: c7e_11. Legal and regulatory barriers related to property ownership and financin


In [19]:
candidate_vars = [
    "k17", "k3_12", "k2_7", "f14", "h7",
    "k8a", "k8a_1", "k8b_1", "g9_12", "l8_2", "m9_4"
]

for col in candidate_vars:
    print(f"\n--- {col}: {labels.get(col, '')} ---")
    print("Non-missing:", df[col].notna().sum())
    print(df[col].value_counts(dropna=False).head(8).to_dict())


--- k17: K17. What is the main reason why you do not have a bank account? ---
Non-missing: 11080
{'Does not need it - Insufficient or no money coming it to justify it': 9204, nan: 2914, 'Other specify': 603, 'Banks are too far away': 336, 'Cannot maintain the minimum balance': 325, 'Bank service charges are too high': 153, 'Does not understand benefits from having a bank account': 120, 'Can get services needed elsewhere in the community': 98}

--- k3_12: K3.12 What is the main reason why do you not have a bank account? ---
Non-missing: 9807
{'Does not need it - Insufficient or no money coming it to justify it': 8170, nan: 4187, 'Other specify': 540, 'Cannot maintain the minimum balance': 264, 'Does not understand benefits of having a UMURENGE SACCO account': 205, 'Can get services needed elsewhere in the community': 125, 'UMURENGE SACCO are too far away': 105, 'Does not have the documentation required': 94}

--- k2_7: K2.7 What is the main reason why do you not have an MFI account? --

In [20]:
pattern = (
    r"mobile money.*(sent|received|transfer|transaction|pay)|"
    r"(sent|received).*money|"
    r"digit(al|ally).*pay|"
    r"pay.*electronically|"
    r"mobile.*transaction"
)

matches = [
    (col, labels.get(col, ""))
    for col in metadata.column_names
    if re.search(pattern, labels.get(col, ""), re.IGNORECASE)
]

print(f"Matching variables: {len(matches)}")
for col, label in matches[:20]:
    print(f"{col}: {label}")
if len(matches) > 20:
    print(f"... {len(matches) - 20} more omitted.")

Matching variables: 42
qf5_01_6: qf5_01_6. Commercial bank-Sent money to, or received money from someone
qf5_01_7: qf5_01_7. Commercial bank-Received money from the government
qf5_02_5: qf5_02_5. Digital bank-Paid a tax, fine, fee, or other payment to the government
qf5_02_6: qf5_02_6. Digital bank-Sent money to, or received money from someone
qf5_02_7: qf5_02_7. Digital bank-Received money from the government
qf5_02_9: qf5_02_9. Digital bank-Made insurance payments or received claims on insurance
qf5_02_14: qf5_02_14. Digital bank-Made pension payment or retirement savings
qf5_03_6: qf5_03_6. Microfinance institution (non-umurenge)-Sent money to, or received mon
qf5_03_7: qf5_03_7. Microfinance institution (non-umurenge)-Received money from the govern
qf5_04_6: qf5_04_6. Non-Deposit Taking Lending Financial (NDFI) -Sent money to, or receive
qf5_04_7: qf5_04_7. Non-Deposit Taking Lending Financial (NDFI) -Received money from the g
qf5_05_5: qf5_05_5. Mobile money operator-Paid a tax, f

In [21]:
candidate_vars = [
    "qf5_05_5",  # Government payments
    "qf5_05_6",  # Transfers
    "qf5_05_7",  # Government receipts
    "qf5_05_8",  # Wage receipts
    "qf5_05_9",  # Insurance payments/claims
    "qf5_05_14", # Pension/retirement payments
]

for col in candidate_vars:
    print(f"\n--- {col}: {labels.get(col, '')} ---")
    print(df[col].value_counts(dropna=False).head(5).to_dict())


--- qf5_05_5: qf5_05_5. Mobile money operator-Paid a tax, fine, fee, or other payment to the g ---
{nan: 9079, 'No': 4638, 'Yes': 277}

--- qf5_05_6: qf5_05_6. Mobile money operator-Sent money to, or received money from someone ---
{nan: 9079, 'Yes': 2578, 'No': 2337}

--- qf5_05_7: qf5_05_7. Mobile money operator-Received money from the government ---
{nan: 9079, 'No': 4792, 'Yes': 123}

--- qf5_05_8: qf5_05_8. Mobile money operator-Received wages from an employer ---
{nan: 9079, 'No': 4777, 'Yes': 138}

--- qf5_05_9: qf5_05_9. Mobile money operator-Made insurance payments or received claims on in ---
{nan: 9079, 'No': 4852, 'Yes': 63}

--- qf5_05_14: qf5_05_14. Mobile money operator-Made pension payment or retirement savings ---
{nan: 9079, 'No': 4903, 'Yes': 12}


In [22]:
pd.crosstab(
    df["f5"],
    df["qf5_05_6"],
    margins=True,
    dropna=False
)

qf5_05_6,No,Yes,All
f5,,,
You have not heard about the Mobile Money,19,7,959
You are not registered and have never used mobile money services,0,0,1877
You are registered but you have never used mobile money services,36,11,452
You are registered and have used it before but not any more,102,68,883
"You use mobile money services, but you are not registered",57,68,389
You are a registered mobile money user,2123,2424,9434
All,2337,2578,13994


In [23]:
for col in ["qf1_05", "qf4_05"]:
    print(f"\n--- {col}: {labels.get(col, 'No label')} ---")
    print(df[col].value_counts(dropna=False).to_dict())

print("\n--- Transfer activity by current mobile-money use ---")
print(pd.crosstab(
    df["qf4_05"],
    df["qf5_05_6"],
    margins=True,
    dropna=False
))


--- qf1_05: qf1_05. Ever used - Mobile money operator ---
{'No': 9079, 'Yes': 4915}

--- qf4_05: qf4_05. Do you currently use - Mobile money operator ---
{nan: 9079, 'Yes': 4720, 'No': 195}

--- Transfer activity by current mobile-money use ---
qf5_05_6    No   Yes    All
qf4_05                     
NaN          0     0   9079
Yes       2207  2513   4720
No         130    65    195
All       2337  2578  13994


In [24]:
pattern = r"save.*(bank|sacco|MFI|financial institution|formal)|" \
          r"(bank|sacco|MFI).*saving|saving.*account"

matches = [
    (col, labels.get(col, ""))
    for col in metadata.column_names
    if re.search(pattern, labels.get(col, ""), re.IGNORECASE)
]

print(f"Matching variables: {len(matches)}")
for col, label in matches[:20]:
    print(f"{col}: {label}")
if len(matches) > 20:
    print(f"... {len(matches) - 20} more omitted.")

Matching variables: 26
qf5_01_14: qf5_01_14. Commercial bank-Made pension payment or retirement savings
qf5_02_14: qf5_02_14. Digital bank-Made pension payment or retirement savings
qf5_06_14: qf5_06_14. Umurenge SACCO-Made pension payment or retirement savings
qf6_09_5: qf6_09_5.Long-Term Saving Scheme (EJOHEZA)-Inability to access bank account thro
uf1_09_1: f1_09_1.use Long-Term Saving Scheme (EJOHEZA)'s  account - belong to someone els
uf1_09_2: f1_09_2.use Long-Term Saving Scheme (EJOHEZA)'s  account - belong to a group acc
uf1_10_1: f1_10_1.use Savings group/ borrow from's account - belong to someone else
uf1_10_2: f1_10_2.use Savings group/ borrow from's account - belong to a group account
uf1_11_1: f1_11_1.use Savings Village/community association's account - belong to someone
uf1_11_2: f1_11_2.use Savings Village/community association's account - belong to a group
uf3_09_12: uf3_09. Documents required when opened: Bank account details_Long-Term Saving Sc
uf3_10_12: uf3_10. Doc

In [25]:
candidate_vars = [
    "uf4_01_1", "uf4_02_1", "uf4_03_1",
    "uf4_04_1", "uf4_05_1", "uf4_06_1",
    "uf4_07_1", "uf4_08_1"
]

for col in candidate_vars:
    print(f"\n--- {col}: {labels.get(col, '')} ---")
    print(df[col].value_counts(dropna=False).head(5).to_dict())


--- uf4_01_1: uf4. Savings accounts with-Commercial bank ---
{nan: 11417, 'No': 1806, 'Yes': 771}

--- uf4_02_1: uf4. Savings accounts with-Digital bank ---
{nan: 13946, 'No': 36, 'Yes': 12}

--- uf4_03_1: uf4. Savings accounts with-Microfinance institution ---
{nan: 13158, 'No': 598, 'Yes': 238}

--- uf4_04_1: uf4. Savings accounts with-Non Deposit Taking Lending Financial (NDFI) ---
{nan: 13940, 'No': 30, 'Yes': 24}

--- uf4_05_1: uf4. Savings accounts with-Mobile money operatorr ---
{nan: 9079, 'No': 3744, 'Yes': 1171}

--- uf4_06_1: uf4. Savings accounts with-Umurenge SACCO ---
{nan: 9822, 'No': 2976, 'Yes': 1196}

--- uf4_07_1: uf4. Savings accounts with-Insurance provider ---
{nan: 13796, 'Yes': 116, 'No': 82}

--- uf4_08_1: uf4. Savings accounts with-Pension fund administrator ---
{nan: 13766, 'Yes': 181, 'No': 47}


In [26]:
for col in ["uf4_01_1", "uf4_05_1", "uf4_06_1", "uf4_03_1"]:
    print(f"\n--- {col} ---")
    print(labels.get(col, "No label"))
    print(df[col].value_counts(dropna=False).to_dict())

print("\nRelated questionnaire labels:")
for col in metadata.column_names:
    label = labels.get(col, "")
    if col.startswith(("uf4_", "uf5_")):
        print(f"{col}: {label}")


--- uf4_01_1 ---
uf4. Savings accounts with-Commercial bank
{nan: 11417, 'No': 1806, 'Yes': 771}

--- uf4_05_1 ---
uf4. Savings accounts with-Mobile money operatorr
{nan: 9079, 'No': 3744, 'Yes': 1171}

--- uf4_06_1 ---
uf4. Savings accounts with-Umurenge SACCO
{nan: 9822, 'No': 2976, 'Yes': 1196}

--- uf4_03_1 ---
uf4. Savings accounts with-Microfinance institution
{nan: 13158, 'No': 598, 'Yes': 238}

Related questionnaire labels:
uf4_01_1: uf4. Savings accounts with-Commercial bank
uf4_01_2: uf4. Current accounts with-Commercial bank
uf4_01_3: uf4. Fixed deposits with-Commercial bank
uf4_01_4: uf4. Outstanding loans with-Commercial bank
uf4_01_5: uf4. Other with-Commercial bank
uf4_02_1: uf4. Savings accounts with-Digital bank
uf4_02_2: uf4. Current accounts with-Digital bank
uf4_02_3: uf4. Fixed deposits with-Digital bank
uf4_02_4: uf4. Outstanding loans with-Digital bank
uf4_02_5: uf4. Other with-Digital bank
uf4_03_1: uf4. Savings accounts with-Microfinance institution
uf4_03_2: 

In [27]:
for col in metadata.column_names:
    label = labels.get(col, "")
    if re.search(
        r"uf4|financial products|which products|products do you|"
        r"financial services.*provider",
        label,
        re.IGNORECASE
    ):
        print(f"{col}: {label}")

e15_1: E15_1. Do you feel that the information on financial products or services bought
e15_6_4: e15_6_4. Fees and charges related to financial products
e15_6_5: e15_6_5. Terms and conditions of financial products
uf4_01_1: uf4. Savings accounts with-Commercial bank
uf4_01_2: uf4. Current accounts with-Commercial bank
uf4_01_3: uf4. Fixed deposits with-Commercial bank
uf4_01_4: uf4. Outstanding loans with-Commercial bank
uf4_01_5: uf4. Other with-Commercial bank
uf4_02_1: uf4. Savings accounts with-Digital bank
uf4_02_2: uf4. Current accounts with-Digital bank
uf4_02_3: uf4. Fixed deposits with-Digital bank
uf4_02_4: uf4. Outstanding loans with-Digital bank
uf4_02_5: uf4. Other with-Digital bank
uf4_03_1: uf4. Savings accounts with-Microfinance institution
uf4_03_2: uf4. Current accounts with-Microfinance institution
uf4_03_3: uf4. Fixed deposits with-Microfinance institution
uf4_03_4: uf4. Outstanding loans with-Microfinance institution
uf4_03_5: uf4. Other with-Microfinance instituti

In [28]:
pattern = (
    r"savings group|saving group|village.*association|"
    r"rotating savings|merry.go.round|"
    r"family.*friends|friends.*family|"
    r"borrowed from.*(family|friend|community)|"
    r"informal.*saving|informal.*borrow"
)

matches = [
    (col, labels.get(col, ""))
    for col in metadata.column_names
    if re.search(pattern, labels.get(col, ""), re.IGNORECASE)
]

print(f"Matching variables: {len(matches)}")
for col, label in matches[:25]:
    print(f"{col}: {label}")
if len(matches) > 25:
    print(f"... {len(matches) - 25} more omitted.")

Matching variables: 159
e18_3: e18_3. Borrowed from saving group
e18_5: e18_5. Borrowed from family/friends/community/church/mosque
e18_9: e18_9. Used savings from a saving group
e18_10: e18_10. Used savings held with friends/family
e18_14: e18_14. Got assistance/gift from friends/family/community(which you did not have
qf1_10: qf1_10. Ever used - Savings group that you save with or borrow from
qf1_11: qf1_11. Ever used - Village/community association
qf2_10: qf2_10. Offer services electronically - Savings group
qf2_11: qf2_11. Offer services electronically - Village/community association
qf3_10: qf3_10. Account registered in own name - Savings group that you save with or bor
qf3_11: qf3_11. Account registered in own name  - Village/community association
qf4_10: qf4_10. Do you currently use - Savings group that you save with or borrow from
qf4_11: qf4_11. Do you currently use - Village/community association
qf5_10_1: qf5_10_1. Savings group that you save with or borrow from-Deposited o

In [29]:
import re
import pandas as pd
import pyreadstat
from pathlib import Path

project_root = Path.cwd().parent
data_path = project_root / "data" / "datasets" / "FinScope2024Rwanda.dta"

# Reload if the notebook kernel was restarted
df = pd.read_stata(data_path, convert_categoricals=True)
_, metadata = pyreadstat.read_dta(data_path, metadataonly=True)
labels = metadata.column_names_to_labels

pattern = r"^qf[1-5]_(10|11)(_|$)|^e18_(3|5|9|10|14)$"

matches = [
    (col, labels.get(col, ""))
    for col in metadata.column_names
    if re.search(pattern, col)
]

print(f"Matching variables: {len(matches)}")

for col, label in matches:
    s = df[col]
    print(f"\n{col}: {label}")
    print(f"Nonmissing: {s.notna().sum():,} | Missing: {s.isna().sum():,}")
    print(s.value_counts(dropna=False).head(5).to_dict())

Matching variables: 45

e18_3: e18_3. Borrowed from saving group
Nonmissing: 11,219 | Missing: 2,775
{'No': 9677, nan: 2775, 'Yes': 1542}

e18_5: e18_5. Borrowed from family/friends/community/church/mosque
Nonmissing: 11,219 | Missing: 2,775
{'No': 11057, nan: 2775, 'Yes': 162}

e18_9: e18_9. Used savings from a saving group
Nonmissing: 11,219 | Missing: 2,775
{'No': 9769, nan: 2775, 'Yes': 1450}

e18_10: e18_10. Used savings held with friends/family
Nonmissing: 11,219 | Missing: 2,775
{'No': 10875, nan: 2775, 'Yes': 344}

e18_14: e18_14. Got assistance/gift from friends/family/community(which you did not have
Nonmissing: 11,219 | Missing: 2,775
{'No': 10939, nan: 2775, 'Yes': 280}

qf1_10: qf1_10. Ever used - Savings group that you save with or borrow from
Nonmissing: 13,994 | Missing: 0
{'Yes': 7026, 'No': 6968}

qf1_11: qf1_11. Ever used - Village/community association
Nonmissing: 13,994 | Missing: 0
{'No': 12828, 'Yes': 1166}

qf2_10: qf2_10. Offer services electronically - Savings

In [30]:
pattern = r"rotating savings|merry.?go.?round|tontine|rosca|informal savings"

for col in metadata.column_names:
    label = labels.get(col, "")
    if re.search(pattern, label, re.IGNORECASE):
        print(f"{col}: {label}")

g3a_11: g3a. Saving with - a group such as a coop, VSLA, tontine, ikibina
h3_09: h3. Borrowed money from a group such as a co-op, VLSA, tontine, ikibina
c7c_3: c7c_3. Borrow from savings group, tontine
c8e_3: c8e_3. Borrow from savings group, tontine


In [31]:
candidate_vars = ["g3a_11", "h3_09", "c7c_3", "c8e_3"]

for col in candidate_vars:
    print(f"\n{col}: {labels.get(col, 'Label unavailable')}")
    print(df[col].value_counts(dropna=False).head(10))


g3a_11: g3a. Saving with - a group such as a coop, VSLA, tontine, ikibina
g3a_11
Yes    7554
No     6440
Name: count, dtype: int64

h3_09: h3. Borrowed money from a group such as a co-op, VLSA, tontine, ikibina
h3_09
No     9126
Yes    4868
Name: count, dtype: int64

c7c_3: c7c_3. Borrow from savings group, tontine
c7c_3
No     8840
NaN    4370
Yes     784
Name: count, dtype: int64

c8e_3: c8e_3. Borrow from savings group, tontine
c8e_3
NaN    9693
No     4002
Yes     299
Name: count, dtype: int64


In [32]:
for col in ["g3a_11", "g3a_17", "h2_1", "h3_09", "c7c_3", "c8e_3"]:
    print(f"{col}: {labels.get(col, 'Label unavailable')}")
    
for col in metadata.column_names:
    label = labels.get(col, "")
    if re.match(r"^(h3|c7c|c8e)(_|\.|$)", label, re.IGNORECASE):
        print(f"{col}: {label}")

g3a_11: g3a. Saving with - a group such as a coop, VSLA, tontine, ikibina
g3a_17: g3a. Saving with - Do not save
h2_1:  H2_1. Borrowed money during the past 12 months
h3_09: h3. Borrowed money from a group such as a co-op, VLSA, tontine, ikibina
c7c_3: c7c_3. Borrow from savings group, tontine
c8e_3: c8e_3. Borrow from savings group, tontine
h3: H3. Please tell me which of the following have you done in the past 12 months?
h3_01: h3. Borrowed money from a bank
h3_02: h3. Borrowed money from Digital loan
h3_03: h3. Borrowed money from a MFI or a non-umurenge SACCO
h3_04: h3. Borrowed money from Non-Deposit Taking Lending Financial (NDFI)
h3_05: h3. Borrowed money through Mobile Money credit e.g MoKash
h3_06: h3. Borrowed from an umurenge SACCO
h3_07: h3. Got a loan from Government (Ubudehe credit scheme/VUP loan)
h3_08: h3. Borrowed money from your employer
h3_09: h3. Borrowed money from a group such as a co-op, VLSA, tontine, ikibina
h3_10: h3. Borrowed money from a money lender in the

In [33]:
import re

# Known mobile-money activity measures
digital_vars = [
    "qf1_05", "qf4_05",
    "qf5_05_6",  # Sent or received money
    "qf5_05_7",  # Government payments
    "qf5_05_8",  # Wages
    "qf5_05_9",  # Insurance payments or claims
]

for col in digital_vars:
    if col in df.columns:
        print(f"\n{col}: {labels.get(col, 'Label unavailable')}")
        print(df[col].value_counts(dropna=False).head(8).to_dict())

print("\n--- Additional digital/payment candidates ---")
pattern = r"mobile money.*(paid|payment|sent|received)|" \
          r"(paid|payment|sent|received).*mobile money|" \
          r"digital payment|electronic payment"

found = 0
for col in metadata.column_names:
    label = labels.get(col, "")
    if re.search(pattern, label, re.IGNORECASE):
        print(f"{col}: {label}")
        found += 1
        if found >= 25:
            print("Output capped at 25 matches.")
            break


qf1_05: qf1_05. Ever used - Mobile money operator
{'No': 9079, 'Yes': 4915}

qf4_05: qf4_05. Do you currently use - Mobile money operator
{nan: 9079, 'Yes': 4720, 'No': 195}

qf5_05_6: qf5_05_6. Mobile money operator-Sent money to, or received money from someone
{nan: 9079, 'Yes': 2578, 'No': 2337}

qf5_05_7: qf5_05_7. Mobile money operator-Received money from the government
{nan: 9079, 'No': 4792, 'Yes': 123}

qf5_05_8: qf5_05_8. Mobile money operator-Received wages from an employer
{nan: 9079, 'No': 4777, 'Yes': 138}

qf5_05_9: qf5_05_9. Mobile money operator-Made insurance payments or received claims on in
{nan: 9079, 'No': 4852, 'Yes': 63}

--- Additional digital/payment candidates ---
qf5_05_2: qf5_05_2. Mobile money operator-Bought airtime top-ups or paid a mobile phone bi
qf5_05_3: qf5_05_3. Mobile money operator-Paid school fees
qf5_05_4: qf5_05_4. Mobile money operator-Paid a bill for medical treatment, housing, tras
qf5_05_5: qf5_05_5. Mobile money operator-Paid a tax, fine,

In [35]:
# Provider-level savings-account product measures
for col in metadata.column_names:
    label = labels.get(col, "")
    if re.match(r"^uf4_\d+_1$", col):
        print(f"\n{col}: {label}")
        print(df[col].value_counts(dropna=False).head(5).to_dict())


uf4_01_1: uf4. Savings accounts with-Commercial bank
{nan: 11417, 'No': 1806, 'Yes': 771}

uf4_02_1: uf4. Savings accounts with-Digital bank
{nan: 13946, 'No': 36, 'Yes': 12}

uf4_03_1: uf4. Savings accounts with-Microfinance institution
{nan: 13158, 'No': 598, 'Yes': 238}

uf4_04_1: uf4. Savings accounts with-Non Deposit Taking Lending Financial (NDFI)
{nan: 13940, 'No': 30, 'Yes': 24}

uf4_05_1: uf4. Savings accounts with-Mobile money operatorr
{nan: 9079, 'No': 3744, 'Yes': 1171}

uf4_06_1: uf4. Savings accounts with-Umurenge SACCO
{nan: 9822, 'No': 2976, 'Yes': 1196}

uf4_07_1: uf4. Savings accounts with-Insurance provider
{nan: 13796, 'Yes': 116, 'No': 82}

uf4_08_1: uf4. Savings accounts with-Pension fund administrator
{nan: 13766, 'Yes': 181, 'No': 47}

uf4_09_1: uf4. Savings accounts with-Long Term Saving Scheme (EJOHEZA)
{nan: 12271, 'Yes': 1420, 'No': 303}

uf4_10_1: uf4. Savings accounts with-Savings group that you save with or borrow from
{nan: 6968, 'Yes': 4659, 'No': 236

In [36]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

candidates = [
    # Existing dashboard indicators
    {
        "Variable": "k1",
        "Indicator": "Owns a bank account",
        "Role": "Core",
        "Status": "Existing baseline",
        "Denominator": "Full eligible survey population; verify valid responses",
        "Decision": "Retain as a main indicator"
    },
    {
        "Variable": "f5",
        "Indicator": "Registered mobile-money user",
        "Role": "Core",
        "Status": "Existing baseline",
        "Denominator": "Full survey population",
        "Decision": "Retain; distinguish registration from current activity"
    },
    {
        "Variable": "h2_1",
        "Indicator": "Borrowed in past 12 months",
        "Role": "Core",
        "Status": "Existing baseline",
        "Denominator": "Full eligible survey population; verify valid responses",
        "Decision": "Retain; borrowing is not automatically positive or negative"
    },
    {
        "Variable": "g3a_17",
        "Indicator": "Does not save",
        "Role": "Core",
        "Status": "Existing baseline",
        "Denominator": "Full survey population",
        "Decision": "Retain; interpret alongside specific saving mechanisms"
    },

    # Priority candidates for extension
    {
        "Variable": "e9b",
        "Indicator": "Has emergency funds",
        "Role": "Supporting",
        "Status": "Responses observed; weighted national rate checked",
        "Denominator": "Full survey population",
        "Decision": "Keep; measures one aspect of financial resilience"
    },
    {
        "Variable": "g3a_11",
        "Indicator": "Saves through a group, VSLA or tontine",
        "Role": "Supporting",
        "Status": "Yes/No responses observed",
        "Denominator": "Full survey population",
        "Decision": "Keep; distinct from not saving at all"
    },
    {
        "Variable": "h3_09",
        "Indicator": "Borrowed from an informal group in past 12 months",
        "Role": "Supporting",
        "Status": "Question wording and period verified",
        "Denominator": "Full survey population; validate coding and routing",
        "Decision": "Keep; source-specific context for h2_1"
    },
    {
        "Variable": "qf4_05",
        "Indicator": "Currently uses mobile-money operator",
        "Role": "Supporting",
        "Status": "Routing observed; denominator validation pending",
        "Denominator": "4,915 ever-users; 9,079 skipped",
        "Decision": "Keep as a candidate; do not treat skipped responses as No"
    },
    {
        "Variable": "qf5_05_6",
        "Indicator": "Sent or received money via mobile money",
        "Role": "Supporting",
        "Status": "Routing observed; denominator validation pending",
        "Denominator": "4,915 ever-users; 9,079 skipped",
        "Decision": "Keep as transaction-activity candidate"
    },
    {
        "Variable": "qf4_10",
        "Indicator": "Currently uses a savings group",
        "Role": "Supporting",
        "Status": "Routing observed; denominator validation pending",
        "Denominator": "7,026 ever-users; 6,968 skipped",
        "Decision": "Keep; compare with direct group-saving measure"
    },
    {
        "Variable": "qf4_11",
        "Indicator": "Currently uses village/community association",
        "Role": "Supporting",
        "Status": "Routing observed; denominator validation pending",
        "Denominator": "1,166 ever-users; 12,828 skipped",
        "Decision": "Keep provisionally; assess district reliability"
    },
    {
        "Variable": "qf4_07",
        "Indicator": "Currently uses an insurance provider",
        "Role": "Supporting",
        "Status": "Sparse responses; routing observed",
        "Denominator": "198 ever-users; 13,796 skipped",
        "Decision": "Retain for context; not a priority district indicator yet"
    },
    {
        "Variable": "e18_1",
        "Indicator": "Borrowed from a bank/formal institution",
        "Role": "Supporting",
        "Status": "Eligibility and routing need validation",
        "Denominator": "11,219 observed; 2,775 missing",
        "Decision": "Keep provisionally; establish reference period first"
    },
    {
        "Variable": "uf4_09_1",
        "Indicator": "EJOHEZA savings account",
        "Role": "Supporting",
        "Status": "Sparse/routed product responses",
        "Denominator": "1,723 observed; 12,271 missing",
        "Decision": "Keep as an exploratory product measure"
    },

    # Candidates requiring more validation
    {
        "Variable": "k17 / f14 / h7",
        "Indicator": "Barriers to bank, mobile-money and borrowing access",
        "Role": "Supporting",
        "Status": "Eligible populations and question routing need validation",
        "Denominator": "Varies by question",
        "Decision": "Keep as separate barrier families; do not combine"
    },
    {
        "Variable": "uf4_*_1",
        "Indicator": "Savings products by provider",
        "Role": "Supporting",
        "Status": "Provider-specific missingness/routing unresolved",
        "Denominator": "Varies by provider",
        "Decision": "Use selected products only after routing validation"
    },

    # Excluded/deferred candidates
    {
        "Variable": "c7c_3 / c8e_3",
        "Indicator": "Borrowing from savings group to finance improvements",
        "Role": "Deferred",
        "Status": "Question purpose identified",
        "Denominator": "Specific improvement-financing population",
        "Decision": "Exclude from general financial-inclusion indicators"
    },
    {
        "Variable": "qf5_05_9",
        "Indicator": "Mobile-money insurance payments/claims",
        "Role": "Deferred",
        "Status": "Only 63 Yes responses observed",
        "Denominator": "Ever-users; routing applies",
        "Decision": "Too sparse for a priority district indicator at present"
    },
]

register = pd.DataFrame(candidates)

register_path = output_dir / "indicator_candidate_register.csv"
register.to_csv(register_path, index=False)

print(f"Saved: {register_path}")
print(f"Total candidates: {len(register)}")
print("\nCandidates by role:")
print(register["Role"].value_counts())
display(register)


Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/indicator_candidate_register.csv
Total candidates: 18

Candidates by role:
Role
Supporting    12
Core           4
Deferred       2
Name: count, dtype: int64


,Variable,Indicator,Role,Status,Denominator,Decision
0,k1,Owns a bank account,Core,Existing baseline,Full eligible survey population; verify valid ...,Retain as a main indicator
1,f5,Registered mobile-money user,Core,Existing baseline,Full survey population,Retain; distinguish registration from current ...
2,h2_1,Borrowed in past 12 months,Core,Existing baseline,Full eligible survey population; verify valid ...,Retain; borrowing is not automatically positiv...
3,g3a_17,Does not save,Core,Existing baseline,Full survey population,Retain; interpret alongside specific saving me...
4,e9b,Has emergency funds,Supporting,Responses observed; weighted national rate che...,Full survey population,Keep; measures one aspect of financial resilience
5,g3a_11,"Saves through a group, VSLA or tontine",Supporting,Yes/No responses observed,Full survey population,Keep; distinct from not saving at all
6,h3_09,Borrowed from an informal group in past 12 months,Supporting,Question wording and period verified,Full survey population; validate coding and ro...,Keep; source-specific context for h2_1
7,qf4_05,Currently uses mobile-money operator,Supporting,Routing observed; denominator validation pending,"4,915 ever-users; 9,079 skipped",Keep as a candidate; do not treat skipped resp...
8,qf5_05_6,Sent or received money via mobile money,Supporting,Routing observed; denominator validation pending,"4,915 ever-users; 9,079 skipped",Keep as transaction-activity candidate
9,qf4_10,Currently uses a savings group,Supporting,Routing observed; denominator validation pending,"7,026 ever-users; 6,968 skipped",Keep; compare with direct group-saving measure


In [37]:

import pandas as pd
import numpy as np

weight_col = "pop_wt"

variables = [
    "k1", "f5", "h2_1", "g3a_17",
    "e9b", "g3a_11", "h3_09",
    "qf4_05", "qf5_05_6",
    "qf4_10", "qf4_11", "qf4_07",
    "e18_1", "uf4_09_1",
]

rows = []

for col in variables:
    if col not in df.columns:
        continue

    s = df[col]
    valid = s.notna()
    weights = df.loc[valid, weight_col]

    rows.append({
        "Variable": col,
        "Question label": labels.get(col, ""),
        "Valid responses": int(valid.sum()),
        "Missing responses": int(s.isna().sum()),
        "Valid response (%)": round(100 * valid.mean(), 2),
        "Weighted valid coverage (%)": round(
            100 * weights.sum() / df[weight_col].sum(), 2
        ),
        "Observed categories": str(s.dropna().unique().tolist()[:8]),
    })

validation = pd.DataFrame(rows)

display(validation)

validation_path = output_dir / "indicator_denominator_validation.csv"
validation.to_csv(validation_path, index=False)
print(f"\nSaved: {validation_path}")


,Variable,Question label,Valid responses,Missing responses,Valid response (%),Weighted valid coverage (%),Observed categories
0,k1,K1. you have a bank account in your name?,13994,0,100.00,100.00,"['Yes', 'No']"
1,f5,F5. Which of the following statements best des...,13994,0,100.00,100.00,"['You are a registered mobile money user', 'Yo..."
2,h2_1,H2_1. Borrowed money during the past 12 months,13994,0,100.00,100.00,"['Yes', 'No']"
3,g3a_17,g3a. Saving with - Do not save,13994,0,100.00,100.00,"['No', 'Yes']"
4,e9b,E9b.Have you set aside emergency/rainy-day fun...,13994,0,100.00,100.00,"['No', 'Yes']"
5,g3a_11,"g3a. Saving with - a group such as a coop, VSL...",13994,0,100.00,100.00,"['Yes', 'No']"
6,h3_09,h3. Borrowed money from a group such as a co-o...,13994,0,100.00,100.00,"['No', 'Yes']"
7,qf4_05,qf4_05. Do you currently use - Mobile money op...,4915,9079,35.12,35.50,"['Yes', 'No']"
8,qf5_05_6,"qf5_05_6. Mobile money operator-Sent money to,...",4915,9079,35.12,35.50,"['No', 'Yes']"
9,qf4_10,qf4_10. Do you currently use - Savings group t...,7026,6968,50.21,48.15,"['Yes', 'No']"



Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/indicator_denominator_validation.csv


In [38]:
for col in ["e9b", "g3a_11"]:
    print(f"\n===== {col} =====")
    print("Question:", labels.get(col, "No variable label found"))
    print(df[col].value_counts(dropna=False).to_string())


===== e9b =====
Question: E9b.Have you set aside emergency/rainy-day funds that would cover your expenses
e9b
No     10976
Yes     3018

===== g3a_11 =====
Question: g3a. Saving with - a group such as a coop, VSLA, tontine, ikibina
g3a_11
Yes    7554
No     6440


In [40]:

import pandas as pd
import numpy as np

weight = pd.to_numeric(df["pop_wt"], errors="coerce")

new_indicators = {
    "Has emergency funds (%)": "e9b",
    "Saves with a group (%)": "g3a_11",
}

def weighted_yes_percentage(data, variable):
    values = data[variable].astype("string").str.strip()
    valid = values.isin(["Yes", "No"]) & data["pop_wt"].notna()
    if valid.sum() == 0:
        return np.nan

    yes = values[valid].eq("Yes").astype(float)
    weights = pd.to_numeric(data.loc[valid, "pop_wt"], errors="coerce")
    valid_weights = weights.notna() & (weights > 0)

    if valid_weights.sum() == 0:
        return np.nan

    return (
        100 * (yes[valid_weights] * weights[valid_weights]).sum()
        / weights[valid_weights].sum()
    )

# National estimates
national_new = pd.DataFrame([
    {
        "Indicator": name,
        "Estimate (%)": weighted_yes_percentage(df, variable),
        "Source variable": variable,
    }
    for name, variable in new_indicators.items()
])

display(national_new.round(2))


# District estimates — one row per district
district_groups = df.groupby("a2", observed=True)

district_rows = []

for district, group in district_groups:
    row = {"District": district}

    for name, variable in new_indicators.items():
        row[name] = weighted_yes_percentage(group, variable)

    district_rows.append(row)

district_new = pd.DataFrame(district_rows)

display(district_new.round(2))
print("Districts:", len(district_new))


,Indicator,Estimate (%),Source variable
0,Has emergency funds (%),20.78,e9b
1,Saves with a group (%),51.83,g3a_11


,District,Has emergency funds (%),Saves with a group (%)
0,Nyarugenge,22.97,37.13
1,Gasabo,26.29,40.76
2,Kicukiro,26.65,38.96
3,Nyanza,10.16,58.81
4,Gisagara,10.81,64.49
5,Nyaruguru,4.76,41.10
6,Huye,26.88,53.68
7,Nyamagabe,31.88,56.49
8,Ruhango,11.20,56.92
9,Muhanga,19.90,53.07


Districts: 30


In [41]:

from pathlib import Path
import pandas as pd

processed = project_root / "data" / "processed"

# 1. Update national indicators
national_path = processed / "national_financial_inclusion.csv"
national = pd.read_csv(national_path)

new_national_rows = pd.DataFrame([
    {
        "Indicator": "Has emergency funds",
        "Estimate (%)": 20.78,
        "Source variable": "e9b",
        "Interpretation": "Respondents who report setting aside emergency or rainy-day funds."
    },
    {
        "Indicator": "Saves with a group",
        "Estimate (%)": 51.83,
        "Source variable": "g3a_11",
        "Interpretation": "Respondents who report saving with a group such as a cooperative, VSLA, tontine, or ikibina."
    }
])

national = national[
    ~national["Indicator"].isin(new_national_rows["Indicator"])
]
national = pd.concat([national, new_national_rows], ignore_index=True)
national.to_csv(national_path, index=False)

# 2. Merge the new district indicators into the existing district file
district_path = processed / "district_financial_inclusion.csv"
district_existing = pd.read_csv(district_path)

district_existing["District"] = district_existing["District"].astype(str)
district_new["District"] = district_new["District"].astype(str)

new_columns = [
    "Has emergency funds (%)",
    "Saves with a group (%)"
]

district_existing = district_existing.drop(
    columns=[c for c in new_columns if c in district_existing.columns]
)

district_updated = district_existing.merge(
    district_new[["District"] + new_columns],
    on="District",
    how="left",
    validate="one_to_one"
)

district_updated.to_csv(district_path, index=False)

# 3. Verify the saved results
print("National indicators saved:", national_path)
print("District indicators saved:", district_path)
print("District count:", len(district_updated))
print("Missing district values:\n",
      district_updated[new_columns].isna().sum())

display(national.tail(2))
display(district_updated[
    ["District"] + new_columns
].head())


National indicators saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/national_financial_inclusion.csv
District indicators saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/district_financial_inclusion.csv
District count: 30
Missing district values:
 Has emergency funds (%)    0
Saves with a group (%)     0
dtype: int64


,Indicator,Estimate (%),Source variable,Interpretation
4,Has emergency funds,20.78,e9b,Respondents who report setting aside emergency...
5,Saves with a group,51.83,g3a_11,Respondents who report saving with a group suc...


,District,Has emergency funds (%),Saves with a group (%)
0,Bugesera,42.613522,66.168455
1,Burera,18.980153,53.636876
2,Gakenke,19.771772,52.691972
3,Gasabo,26.285342,40.764305
4,Gatsibo,26.785999,66.063032


In [42]:
# Search variable names and questionnaire labels for the next candidates.
keywords = [
    "mobile money",
    "actively use",
    "use mobile",
    "save",
    "savings",
    "SACCO",
    "credit",
    "borrow",
    "insurance",
    "reason",
    "barrier",
]

for keyword in keywords:
    matches = [
        (col, str(label))
        for col, label in labels.items()
        if keyword.lower() in str(label).lower()
    ]

    print(f"\n{'=' * 20} {keyword.upper()} {'=' * 20}")
    for col, label in matches[:30]:
        print(f"{col}: {label}")
    print("Matches shown:", min(len(matches), 30), "| Total:", len(matches))


==================== MOBILE MONEY ====================
qf1_05: qf1_05. Ever used - Mobile money operator
qf2_05: qf2_05. Offer services electronically - Mobile money operator
qf3_05: qf3_05. Account registered in own name - Mobile money operator
qf4_05: qf4_05. Do you currently use - Mobile money operator
qf5_05_1: qf5_05_1. Mobile money operator-Deposited or withdrew money
qf5_05_2: qf5_05_2. Mobile money operator-Bought airtime top-ups or paid a mobile phone bi
qf5_05_3: qf5_05_3. Mobile money operator-Paid school fees
qf5_05_4: qf5_05_4. Mobile money operator-Paid a bill for medical treatment, housing, tras
qf5_05_5: qf5_05_5. Mobile money operator-Paid a tax, fine, fee, or other payment to the g
qf5_05_6: qf5_05_6. Mobile money operator-Sent money to, or received money from someone
qf5_05_7: qf5_05_7. Mobile money operator-Received money from the government
qf5_05_8: qf5_05_8. Mobile money operator-Received wages from an employer
qf5_05_9: qf5_05_9. Mobile money operator-Made insu

In [43]:
candidates = ["qf4_05", "qf4_07", "e18_1", "k17"]

for col in candidates:
    print(f"\n{'=' * 55}")
    print(f"VARIABLE: {col}")
    print("QUESTION:", labels.get(col, "No label found"))
    print("RESPONSE COUNTS:")
    print(df[col].value_counts(dropna=False).to_string())


VARIABLE: qf4_05
QUESTION: qf4_05. Do you currently use - Mobile money operator
RESPONSE COUNTS:
qf4_05
NaN    9079
Yes    4720
No      195

VARIABLE: qf4_07
QUESTION: qf4_07. Do you currently use - Insurance provider
RESPONSE COUNTS:
qf4_07
NaN    13796
Yes      175
No        23

VARIABLE: e18_1
QUESTION: e18_1. Borrowed from a bank or other formal institution
RESPONSE COUNTS:
e18_1
No     10411
NaN     2775
Yes      808

VARIABLE: k17
QUESTION: K17. What is the main reason why you do not have a bank account?
RESPONSE COUNTS:
k17
Does not need it - Insufficient or no money coming it to justify it    9204
NaN                                                                    2914
Other specify                                                           603
Banks are too far away                                                  336
Cannot maintain the minimum balance                                     325
Bank service charges are too high                                       153
Does n

In [44]:
# Check whether current mobile-money use agrees with f5
print("MOBILE MONEY: qf4_05 vs f5")
print(pd.crosstab(df["qf4_05"], df["f5"], margins=True, dropna=False))

# Check formal borrowing against borrowing from any source
print("\nFORMAL BORROWING: e18_1 vs h2_1")
print(pd.crosstab(df["e18_1"], df["h2_1"], margins=True, dropna=False))

# Check bank-account ownership against the reasons for not having an account
print("\nBANK OWNERSHIP: k1 vs k17")
print(pd.crosstab(df["k1"], df["k17"], margins=True, dropna=False))

MOBILE MONEY: qf4_05 vs f5
f5      You have not heard about the Mobile Money  \
qf4_05                                              
NaN                                           933   
Yes                                            19   
No                                              7   
All                                           959   

f5      You are not registered and have never used mobile money services  \
qf4_05                                                                     
NaN                                                  1877                  
Yes                                                     0                  
No                                                      0                  
All                                                  1877                  

f5      You are registered but you have never used mobile money services  \
qf4_05                                                                     
NaN                                          

In [45]:
# Reasons for not having a bank account
reasons = df.loc[(df["k1"] == "No") & df["k17"].notna(), ["k17", "pop_wt"]].copy()

reason_summary = (
    reasons.groupby("k17", observed=True)
    .agg(Respondents=("k17", "size"), Weighted_population=("pop_wt", "sum"))
    .reset_index()
)

reason_summary["Weighted share (%)"] = (
    reason_summary["Weighted_population"]
    / reason_summary["Weighted_population"].sum()
    * 100
)

reason_summary = reason_summary.sort_values("Weighted share (%)", ascending=False)

display(reason_summary.round({"Weighted share (%)": 2}))

,k17,Respondents,Weighted_population,Weighted share (%)
0,Does not need it - Insufficient or no money co...,9204,5.404338e+06,83.04
12,Other specify,603,3.446711e+05,5.30
3,Banks are too far away,336,1.961213e+05,3.01
1,Cannot maintain the minimum balance,325,1.800371e+05,2.77
2,Bank service charges are too high,153,8.908874e+04,1.37
7,Does not understand benefits from having a ban...,120,6.890685e+04,1.06
5,Does not have the documentation required,73,6.352225e+04,0.98
6,Does not know how to apply,90,5.522971e+04,0.85
10,Can get services needed elsewhere in the commu...,98,5.273025e+04,0.81
11,Need permission of someone else to open it,27,2.259650e+04,0.35


In [46]:
from pathlib import Path

output_path = Path("../data/processed/bank_account_barriers.csv")

reason_summary.to_csv(output_path, index=False)

print(f"Saved: {output_path.resolve()}")
print(f"Response categories: {len(reason_summary)}")

Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/bank_account_barriers.csv
Response categories: 13


In [47]:
import pandas as pd
from pathlib import Path

# Respondents without a bank account who reported a reason
barrier_data = df.loc[
    (df["k1"] == "No") & df["k17"].notna(), ["a2", "k17", "pop_wt"]
].copy()

# District-level denominator: all respondents without an account
district_totals = (
    barrier_data.groupby("a2", observed=True)
    .agg(
        **{
            "District non-account respondents": ("k17", "size"),
            "District weighted population": ("pop_wt", "sum"),
        }
    )
    .reset_index()
    .rename(columns={"a2": "District"})
)

# Count and weighted population for each reason within each district
district_barriers = (
    barrier_data.groupby(["a2", "k17"], observed=True)
    .agg(
        Respondents=("k17", "size"),
        Weighted_population=("pop_wt", "sum"),
    )
    .reset_index()
    .rename(columns={"a2": "District"})
)

district_barriers = district_barriers.merge(district_totals, on="District", how="left")

district_barriers["Weighted share (%)"] = (
    district_barriers["Weighted_population"]
    / district_barriers["District weighted population"]
    * 100
)

district_barriers = district_barriers.rename(columns={"k17": "Reason"})

district_barriers = district_barriers.sort_values(
    ["District", "Weighted share (%)"], ascending=[True, False]
).reset_index(drop=True)

# Save alongside the existing national file
output_path = Path("../data/processed/district_bank_account_barriers.csv")
district_barriers.to_csv(output_path, index=False)

print(f"Saved: {output_path.resolve()}")
print(f"Districts: {district_barriers['District'].nunique()}")
print(f"District-reason rows: {len(district_barriers)}")
print("Missing weighted shares:", district_barriers["Weighted share (%)"].isna().sum())

display(district_barriers.head(15))

Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/district_bank_account_barriers.csv
Districts: 30
District-reason rows: 294
Missing weighted shares: 0


,District,Reason,Respondents,Weighted_population,District non-account respondents,District weighted population,Weighted share (%)
0,Nyarugenge,Does not need it - Insufficient or no money co...,168,101660.692948,213,123911.495866,82.042987
1,Nyarugenge,Other specify,16,8124.274697,213,123911.495866,6.556514
2,Nyarugenge,Banks are too far away,11,7349.936265,213,123911.495866,5.931602
3,Nyarugenge,Bank service charges are too high,9,4326.224607,213,123911.495866,3.491383
4,Nyarugenge,Cannot maintain the minimum balance,7,1840.649469,213,123911.495866,1.485455
5,Nyarugenge,Does not trust banks,1,337.977379,213,123911.495866,0.272757
6,Nyarugenge,Can get services needed elsewhere in the commu...,1,271.740501,213,123911.495866,0.219302
7,Gasabo,Does not need it - Insufficient or no money co...,366,268207.408491,434,307627.705485,87.185713
8,Gasabo,Other specify,27,16790.848251,434,307627.705485,5.458172
9,Gasabo,Banks are too far away,16,9672.015815,434,307627.705485,3.144065


In [48]:
district_share_check = district_barriers.groupby("District")["Weighted share (%)"].sum()

print("Districts checked:", len(district_share_check))
print("Lowest district total:", district_share_check.min())
print("Highest district total:", district_share_check.max())

print("\nDistricts whose shares do not sum to approximately 100%:")
display(district_share_check[~district_share_check.between(99.99, 100.01)])

Districts checked: 30
Lowest district total: 99.99999999999999
Highest district total: 100.00000000000001

Districts whose shares do not sum to approximately 100%:


Series([], Name: Weighted share (%), dtype: float64)